# GPU and PyTorch Verification Notebook

This notebook verifies that:
1. GPU is properly recognized by the system
2. CUDA is available and configured
3. PyTorch can access the GPU
4. PyTorch computations run on GPU

**Run this notebook after connecting your remote GPU kernel in VS Code.**

## 1. Check System Information

In [ ]:
import os
import platform

print("=" * 60)
print("SYSTEM INFORMATION")
print("=" * 60)
print(f"Platform: {platform.system()} {platform.release()}")
print(f"Python Version: {platform.python_version()}")
print(f"Hostname: {os.environ.get('HOSTNAME', 'Unknown')}")
print(f"SLURM Job ID: {os.environ.get('SLURM_JOB_ID', 'Not in SLURM job')}")
print(f"Current Working Directory: {os.getcwd()}")

## 2. Check GPU Hardware

In [ ]:
import subprocess

print("=" * 60)
print("GPU HARDWARE CHECK (nvidia-smi)")
print("=" * 60)

try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
except FileNotFoundError:
    print("❌ ERROR: nvidia-smi not found. GPU drivers may not be installed.")
except Exception as e:
    print(f"❌ ERROR: {e}")

## 3. Check PyTorch Installation

In [ ]:
import torch

print("=" * 60)
print("PYTORCH INFORMATION")
print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"cuDNN Version: {torch.backends.cudnn.version()}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.current_device()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Properties: {torch.cuda.get_device_properties(0)}")
else:
    print("❌ ERROR: CUDA not available. PyTorch may not be built with CUDA support.")

## 4. Test GPU Memory Access

In [ ]:
if torch.cuda.is_available():
    print("=" * 60)
    print("GPU MEMORY INFORMATION")
    print("=" * 60)
    
    gpu_index = 0
    
    # Get memory info
    total_memory = torch.cuda.get_device_properties(gpu_index).total_memory
    reserved_memory = torch.cuda.memory_reserved(gpu_index)
    allocated_memory = torch.cuda.memory_allocated(gpu_index)
    free_memory = reserved_memory - allocated_memory
    
    print(f"Total GPU Memory: {total_memory / 1e9:.2f} GB")
    print(f"Reserved Memory: {reserved_memory / 1e9:.2f} GB")
    print(f"Allocated Memory: {allocated_memory / 1e9:.2f} GB")
    print(f"Free Memory: {free_memory / 1e9:.2f} GB")
else:
    print("❌ SKIPPED: GPU not available")

## 5. Test Basic GPU Computation

In [ ]:
import time

if torch.cuda.is_available():
    print("=" * 60)
    print("GPU COMPUTATION TEST")
    print("=" * 60)
    
    # Create a large tensor on GPU
    size = 10000
    
    # CPU computation
    print(f"\nCreating {size}x{size} matrix...")
    
    start = time.time()
    cpu_tensor = torch.randn(size, size)
    cpu_result = torch.mm(cpu_tensor, cpu_tensor)
    cpu_time = time.time() - start
    print(f"✓ CPU Computation Time: {cpu_time:.4f} seconds")
    
    # GPU computation
    torch.cuda.synchronize()  # Wait for GPU to be ready
    start = time.time()
    gpu_tensor = torch.randn(size, size, device='cuda')
    gpu_result = torch.mm(gpu_tensor, gpu_tensor)
    torch.cuda.synchronize()  # Wait for GPU to finish
    gpu_time = time.time() - start
    print(f"✓ GPU Computation Time: {gpu_time:.4f} seconds")
    
    speedup = cpu_time / gpu_time
    print(f"\n✓ GPU Speedup: {speedup:.2f}x faster than CPU")
    print("\n✅ GPU is working correctly!")
else:
    print("❌ SKIPPED: GPU not available")

## 6. Test PyTorch Data Types and Device Transfer

In [ ]:
if torch.cuda.is_available():
    print("=" * 60)
    print("DEVICE TRANSFER TEST")
    print("=" * 60)
    
    # Create tensor on CPU
    cpu_x = torch.tensor([1.0, 2.0, 3.0])
    print(f"CPU Tensor: {cpu_x}")
    print(f"Device: {cpu_x.device}")
    
    # Move to GPU
    gpu_x = cpu_x.cuda()
    print(f"\nGPU Tensor: {gpu_x}")
    print(f"Device: {gpu_x.device}")
    
    # Perform computation on GPU
    result = gpu_x * 2 + 1
    print(f"\nComputation Result (on GPU): {result}")
    print(f"Device: {result.device}")
    
    # Move back to CPU
    cpu_result = result.cpu()
    print(f"\nBack to CPU: {cpu_result}")
    print(f"Device: {cpu_result.device}")
    
    print("\n✅ Device transfer working correctly!")
else:
    print("❌ SKIPPED: GPU not available")

## 7. Summary and Verification Status

In [ ]:
print("=" * 60)
print("VERIFICATION SUMMARY")
print("=" * 60)

checks = {
    "Python Installed": True,
    "PyTorch Installed": True,
    "CUDA Available": torch.cuda.is_available(),
    "GPU Detected": torch.cuda.is_available() and torch.cuda.device_count() > 0,
}

for check, status in checks.items():
    symbol = "✅" if status else "❌"
    print(f"{symbol} {check}: {'Yes' if status else 'No'}")

print("\n" + "=" * 60)
if all(checks.values()):
    print("✅ ALL CHECKS PASSED - GPU SETUP IS WORKING!")
    print("You can now run GPU-accelerated notebooks.")
else:
    print("❌ SOME CHECKS FAILED - See the documentation for troubleshooting.")
    print("Reference: HPC-Setup/docs/GPU_NOTEBOOK_SETUP.md")
print("=" * 60)

## Next Steps

If all checks passed:
- ✅ Your GPU setup is working correctly
- ✅ You can now run GPU-accelerated machine learning notebooks
- ✅ Refer to [TEST_GPU_PYTORCH.md](./docs/TEST_GPU_PYTORCH.md) for testing with real models

If some checks failed:
- ❌ Consult the troubleshooting guides in `HPC-Setup/docs/`
- ❌ Start with [GPU_NOTEBOOK_SETUP.md](./docs/GPU_NOTEBOOK_SETUP.md)
- ❌ Check [JUPYTER_KERNEL_NOT_FOUND.md](./docs/JUPYTER_KERNEL_NOT_FOUND.md) for kernel issues

## Documentation References

- **Main Setup Guide**: [GPU_NOTEBOOK_SETUP.md](./docs/GPU_NOTEBOOK_SETUP.md)
- **Quick Start**: [SETUP_README.md](./docs/SETUP_README.md)
- **Documentation Index**: [DOCUMENTATION_INDEX.md](./docs/DOCUMENTATION_INDEX.md)
- **Troubleshooting Guides**: See [docs/](./docs/) folder